# pipe_2 — Preprocesamiento y feature engineering, en un solo paso

Unifica lo que en `src/pipe/` son `01_Preprocesamiento` y `02_FE`, y agrega una familia
de features que el pipe original no tiene: **shares, deltas de share, índices y
promedios móviles**.

La salida es un parquet que `03_Optuna` lee **sin cambios**: lleva el mismo formato de
nombre más un marcador `_fe2`, así los experimentos de los dos pipes conviven en el
mismo leaderboard y son comparables.

## Qué agrega respecto de `02_FE`

| Familia | Qué es |
|---|---|
| **Shares** | La participación de cada fila en cuatro denominadores distintos |
| **Deltas de share** | Cuánto cambió esa participación: contra el mes anterior y contra su propia media móvil |
| **Índices** | Ratio contra el mes anterior: toneladas, cantidad pedida, clientes distintos |
| **Promedios móviles** | De ventas y de shares, a 3, 6 y 12 meses |

## Y qué hace distinto

`02_FE` genera ~616 columnas: 24 lags de todo, en cinco niveles de agregación. Acá el
criterio es otro — **menos features y mejor pensadas**. Los lags de la propia serie ya
están en el pipe original; lo que falta es el *contexto competitivo*, que es lo que
aportan los shares. Con `max_lags=12` esto queda en ~120 columnas en vez de 616:
menos memoria, entrena más rápido, y cada columna tiene una razón para estar.

## El share a nivel cliente-producto

Con granularidad `pc` cada fila es un par producto-cliente, y ahí «share» puede
significar cuatro cosas distintas. Las cuatro se calculan, porque dicen cosas distintas:

| Share | Fórmula | Qué dice |
|---|---|---|
| del producto **en el cliente** | `tn(p,c,t) / tn(c,t)` | qué tan importante es ese producto para ese cliente |
| del cliente **en el producto** | `tn(p,c,t) / tn(p,t)` | qué tan importante es ese cliente para ese producto |
| del producto **en su categoría** | `tn(p,t) / tn(cat_k,t)` | la participación de mercado clásica, en los 3 niveles |
| del producto **en el mercado** | `tn(p,t) / tn(t)` | ídem sin jerarquía |

Con granularidad `p` las dos primeras no existen (no hay dimensión cliente) y se saltean
solas.

## Data leakage: la regla y los controles

**Toda feature de la fila del período `t` usa sólo información disponible hasta `t`
inclusive.** El target es `tn(t + horizonte)`.

Las tres trampas concretas, todas verificadas por la celda de control:

1. **Nada de agregados de toda la serie.** El máximo histórico se calcula expansivo
   (`cum_max`), no global. La duración de la vida y el mes de muerte no se usan.
2. **Los denominadores de los shares son del mes `t`**, que se conoce en `t`. No hay
   que confundir «usa datos de otros productos» con «usa el futuro»: la venta de la
   categoría en `t` es un dato de `t`.
3. **La edad es causal**: `-1` mientras el producto todavía no vendió nunca. Con
   densificación completa hay filas anteriores al lanzamiento, y poner ahí
   `m - m_nace` sería saber en `t` que el producto se lanza en `t+5`.

## 0 — Ambiente

In [ ]:
import gc, json, os, time
from pathlib import Path

import numpy as np
import polars as pl


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET  = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
RUTA_FE = BUCKET / "datasets_fe"      # la misma que el pipe original: 03_Optuna lee de aca
RUTA_FE.mkdir(parents=True, exist_ok=True)

print(f"BUCKET  : {BUCKET}")
print(f"crudos  : {DIR_RAW}")
print(f"salida  : {RUTA_FE}")

## 1 — Palancas

Las de `01_Preprocesamiento` y las de `02_FE` juntas, más las propias de las features
nuevas. Todas entran en el nombre del archivo, así que dos configuraciones nunca se
pisan.

In [ ]:
PARAM = {
    # ══ De 01_Preprocesamiento ═══════════════════════════════════════════
    # 'pc' -> una fila por producto-cliente-mes | 'p' -> una fila por producto-mes
    'granularidad': 'pc',

    # Que pasa con un mes sin venta dentro de la vida de la serie.
    # 'cero' -> vale 0.0, que es lo que fisicamente paso.
    # 'nulo' -> queda vacio y LightGBM lo trata como faltante.
    'faltantes': 'cero',

    # 'vida'  -> cada serie existe entre su primera y su ultima venta.
    # 'total' -> todas las series en todos los meses del rango global.
    #            Necesario si queres fila de inferencia para TODOS los productos
    #            a entregar, incluidos los que dejaron de venderse.
    'densificar': 'vida',

    # Solo los 780 productos que se entregan. True descarta ~35% de las filas sin
    # perder nada de lo que se evalua.
    'solo_productos_target': True,

    # Quedarse con los N productos de mayor volumen. None = todos.
    # Para validar el pipe entero en minutos: queda anotado en el nombre (_smplN).
    'muestra_productos': None,

    # ══ De 02_FE ═════════════════════════════════════════════════════════
    # La fila de t predice tn(t + horizonte). DEBE coincidir con 03_Optuna.
    'horizonte': 2,

    # Lags de la propia serie. 12 en vez de 24: con 12 ya se ve el ciclo anual, y el
    # ancho del dataset es la mitad. Los shares aportan mas por columna que el lag 20.
    'max_lags': 12,

    # Ventanas de los promedios moviles (incluyen el mes actual, son causales).
    'ventanas_ma': (3, 6, 12),

    # ══ Features nuevas ══════════════════════════════════════════════════
    # Niveles de jerarquia sobre los que calcular el share del producto.
    'niveles_share': ('cat1', 'cat2', 'cat3', 'mercado'),

    # Cuantos lags de los shares (para poder calcular deltas contra t-k).
    'lags_share': 3,

    # Recorte de los indices (ratios contra el mes anterior). Un producto que pasa de
    # 0,01 a 10 toneladas da un ratio de 1000, y ese outlier no informa nada: satura
    # los splits del arbol. Se recorta a este techo.
    'techo_indice': 10.0,

    'semilla': 102191,
}

G = PARAM['granularidad']
H = PARAM['horizonte']
L = PARAM['max_lags']
ES_PC = G == 'pc'
KEYS = ['product_id', 'customer_id'] if ES_PC else ['product_id']

# ── Nombre del archivo: arrastra TODAS las palancas ──────────────────────
# Mismo formato que el pipe original para que 03_Optuna lo parsee, mas el marcador
# _fe2 al final. Sin ese marcador, un dataset de pipe_2 pisaria el del pipe original.
_grp = 'grpClienteProducto' if ES_PC else 'grpProducto'
_fill = 'fill0' if PARAM['faltantes'] == 'cero' else 'fillNA'
_dense = 'denseLife' if PARAM['densificar'] == 'vida' else 'denseFull'
_tgt = '_tgtFilter' if PARAM['solo_productos_target'] else ''
_smpl = f"_smpl{PARAM['muestra_productos']}" if PARAM['muestra_productos'] else ''
NOMBRE = (f"preprocesado_{_grp}_{_fill}_{_dense}{_tgt}{_smpl}"
          f"_{L}lags_share_{H}h_fe2.parquet")

print(f"granularidad : {G}   (claves: {KEYS})")
print(f"horizonte    : {H}   lags: {L}")
print(f"niveles share: {PARAM['niveles_share']}")
print(f"\nsalida: {NOMBRE}")

## 2 — Preprocesamiento: el panel

Lee los crudos, arma la grilla de series y la densifica. Es lo mismo que hace
`01_Preprocesamiento`, condensado.

In [ ]:
t0 = time.time()

sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
          .unique(subset=["product_id"]))
target_ids = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt")["product_id"].to_list()

print(f"sell-in: {sell.height:,} filas · {sell['product_id'].n_unique()} productos "
      f"· {sell['customer_id'].n_unique()} clientes")


def a_m(periodo):
    """AAAAMM -> indice de mes continuo, para poder sumar y restar meses."""
    return (periodo // 100) * 12 + (periodo % 100)


def m_a_periodo(m):
    return ((m - 1) // 12) * 100 + ((m - 1) % 12) + 1


# ── Filtros de productos ─────────────────────────────────────────────────
if PARAM['solo_productos_target']:
    antes = sell['product_id'].n_unique()
    sell = sell.filter(pl.col("product_id").is_in(target_ids))
    print(f"solo productos target: {antes} -> {sell['product_id'].n_unique()} productos")

if PARAM['muestra_productos']:
    top = (sell.group_by("product_id").agg(pl.col("tn").sum().alias("tn"))
               .sort("tn", descending=True)
               .head(PARAM['muestra_productos'])["product_id"].to_list())
    sell = sell.filter(pl.col("product_id").is_in(top))
    print(f"muestra: los {len(top)} productos de mayor volumen")

# ── Agregacion a la granularidad elegida ─────────────────────────────────
panel = (sell.group_by(KEYS + ["periodo"])
             .agg(pl.col("tn").sum().alias("tn"),
                  pl.col("cust_request_tn").sum().alias("req_tn"),
                  pl.col("cust_request_qty").sum().alias("req_qty"),
                  pl.col("plan_precios_cuidados").max().alias("precios_cuidados"),
                  *([pl.col("customer_id").n_unique().alias("n_clientes")]
                    if not ES_PC else []))
             .with_columns(a_m(pl.col("periodo")).alias("m")))

M_MIN, M_MAX = int(panel["m"].min()), int(panel["m"].max())
print(f"panel: {panel.height:,} filas · rango {m_a_periodo(M_MIN)} -> {m_a_periodo(M_MAX)}")

# ── Densificacion ────────────────────────────────────────────────────────
vida = panel.group_by(KEYS).agg(pl.col("m").min().alias("m_nace"),
                                pl.col("m").max().alias("m_muere"))

if PARAM['densificar'] == 'total':
    grilla = (vida.select(KEYS)
                  .join(pl.DataFrame({"m": list(range(M_MIN, M_MAX + 1))}), how="cross"))
else:
    grilla = (vida.with_columns(
                    pl.int_ranges("m_nace", pl.col("m_muere") + 1).alias("m"))
                  .explode("m").select(KEYS + ["m"]))

_fill = 0.0 if PARAM['faltantes'] == 'cero' else None
panel = grilla.join(panel.drop("periodo"), on=KEYS + ["m"], how="left")
if PARAM['faltantes'] == 'cero':
    panel = panel.with_columns(
        pl.col("tn").fill_null(0.0), pl.col("req_tn").fill_null(0.0),
        pl.col("req_qty").fill_null(0), pl.col("precios_cuidados").fill_null(0),
        *([pl.col("n_clientes").fill_null(0)] if not ES_PC else []))

panel = (panel.join(vida, on=KEYS, how="left")
              .join(prod.select("product_id", "cat1", "cat2", "cat3", "brand", "sku_size"),
                    on="product_id", how="left")
              .with_columns(
                  # vectorizado, NO map_elements: esa es una UDF de Python fila por
                  # fila y sobre 9M filas tarda minutos.
                  ((((pl.col("m") - 1) // 12) * 100)
                   + ((pl.col("m") - 1) % 12) + 1).alias("periodo"),
                  # edad CAUSAL: -1 mientras la serie no vendio nunca. Con densificar
                  # 'total' hay filas previas al lanzamiento, y m - m_nace ahi seria
                  # saber en t que el producto arranca en t+5.
                  pl.when(pl.col("m") >= pl.col("m_nace"))
                    .then(pl.col("m") - pl.col("m_nace"))
                    .otherwise(-1).alias("edad"))
              .sort(KEYS + ["m"]))

print(f"densificado ({PARAM['densificar']}): {panel.height:,} filas")
print(f"ceros de tn: {int((panel['tn'] == 0).sum()):,} "
      f"({100*(panel['tn'] == 0).sum()/panel.height:.0f}%)")
print(f"[{time.time()-t0:.0f}s]")

## 3 — Los denominadores de los shares

Para cada mes se calculan los totales por los que después se divide. **Todos son del
mes `t`**, así que se conocen en `t`: usar la venta de la categoría en `t` no es
leakage, es contexto.

Ojo con un detalle: los totales se calculan sobre el panel **ya filtrado**. Si corriste
con `solo_productos_target=True`, el total de una `cat3` es el de los productos que se
entregan, no el del mercado real. Es coherente — todas las filas usan el mismo
denominador — pero es una decisión, no un descuido: el share pasa a ser «participación
dentro del universo que modelo».

In [ ]:
t0 = time.time()

# ── Total del producto en el mes (suma sobre clientes) ───────────────────
# Con granularidad 'p' el panel YA esta a nivel producto, asi que tn_prod == tn.
if ES_PC:
    tot_prod = (panel.group_by(["product_id", "m"])
                     .agg(pl.col("tn").sum().alias("tn_prod"),
                          pl.len().alias("n_clientes_prod")))
else:
    tot_prod = panel.select(["product_id", "m", pl.col("tn").alias("tn_prod")]) \
                    .with_columns(pl.lit(1).alias("n_clientes_prod"))

# ── Total del cliente en el mes (suma sobre productos) ───────────────────
if ES_PC:
    tot_cli = (panel.group_by(["customer_id", "m"])
                    .agg(pl.col("tn").sum().alias("tn_cli"),
                         pl.len().alias("n_productos_cli")))

# ── Totales de la jerarquia y del mercado, a nivel PRODUCTO ──────────────
# Se calculan sobre tot_prod (una fila por producto-mes) para no contar dos veces
# cuando la granularidad es 'pc'.
prod_mes = tot_prod.join(prod.select("product_id", "cat1", "cat2", "cat3"),
                         on="product_id", how="left")

tot_niveles = {}
for niv in PARAM['niveles_share']:
    if niv == 'mercado':
        t = prod_mes.group_by("m").agg(pl.col("tn_prod").sum().alias("tn_mercado"))
        tot_niveles['mercado'] = t
    else:
        t = (prod_mes.group_by([niv, "m"])
                     .agg(pl.col("tn_prod").sum().alias(f"tn_{niv}"),
                          pl.len().alias(f"n_prod_{niv}")))
        tot_niveles[niv] = t
    print(f"  total {niv}: {tot_niveles[niv].height:,} filas")

print(f"[{time.time()-t0:.0f}s]")

## 4 — Shares

La participación de cada fila en cada denominador. Todas las divisiones pasan por
`div_segura`: si el denominador es 0 el share es 0, no infinito ni nulo.

Un share vale entre 0 y 1 y **es adimensional**, que es justamente su virtud: hace
comparables un producto de 1000 toneladas y uno de 5. Donde `tn` te dice el tamaño, el
share te dice la *posición relativa* — y la posición relativa es lo que se redistribuye
cuando entra un competidor.

In [ ]:
t0 = time.time()


def div_segura(num, den, nombre):
    """num / den con 0 donde el denominador es 0 o nulo. Nunca inf ni nan."""
    return (pl.when(pl.col(den).abs() > 1e-9)
              .then(pl.col(num) / pl.col(den))
              .otherwise(0.0).alias(nombre))


df = panel

# ── Shares propios de la granularidad pc ─────────────────────────────────
if ES_PC:
    df = (df.join(tot_prod, on=["product_id", "m"], how="left")
            .join(tot_cli, on=["customer_id", "m"], how="left")
            .with_columns(
                # que tan importante es este PRODUCTO para este CLIENTE
                div_segura("tn", "tn_cli", "sh_prod_en_cli"),
                # que tan importante es este CLIENTE para este PRODUCTO
                div_segura("tn", "tn_prod", "sh_cli_en_prod"),
            ))
    SHARES = ["sh_prod_en_cli", "sh_cli_en_prod"]
else:
    df = df.join(tot_prod, on=["product_id", "m"], how="left")
    SHARES = []

# ── Shares del producto en la jerarquia y en el mercado ──────────────────
for niv in PARAM['niveles_share']:
    t = tot_niveles[niv]
    if niv == 'mercado':
        df = df.join(t, on="m", how="left")
        df = df.with_columns(div_segura("tn_prod", "tn_mercado", "sh_prod_en_mercado"))
        SHARES.append("sh_prod_en_mercado")
    else:
        df = df.join(t, on=[niv, "m"], how="left")
        df = df.with_columns(div_segura("tn_prod", f"tn_{niv}", f"sh_prod_en_{niv}"))
        SHARES.append(f"sh_prod_en_{niv}")

print(f"{len(SHARES)} shares: {SHARES}")
print(f"[{time.time()-t0:.0f}s]")

# ── Chequeo: los shares del mismo denominador tienen que sumar 1 por grupo ──
# Es el control de que las divisiones estan bien armadas. Se mira un mes cualquiera.
_m_test = int(df["m"].median())
_chk = df.filter(pl.col("m") == _m_test)
print(f"\nchequeo de consistencia en el mes {m_a_periodo(_m_test)}:")
if ES_PC:
    _s = (_chk.group_by("customer_id").agg(pl.col("sh_prod_en_cli").sum().alias("s"))["s"])
    print(f"  suma de sh_prod_en_cli por cliente : min {_s.min():.4f}  max {_s.max():.4f}"
          f"   (deberia ser 1)")
    _s2 = (_chk.group_by("product_id").agg(pl.col("sh_cli_en_prod").sum().alias("s"))["s"])
    print(f"  suma de sh_cli_en_prod por producto: min {_s2.min():.4f}  max {_s2.max():.4f}"
          f"   (deberia ser 1)")
if 'cat3' in PARAM['niveles_share']:
    _s3 = (_chk.unique(subset=["product_id"])
                .group_by("cat3").agg(pl.col("sh_prod_en_cat3").sum().alias("s"))["s"])
    print(f"  suma de sh_prod_en_cat3 por cat3   : min {_s3.min():.4f}  max {_s3.max():.4f}"
          f"   (deberia ser 1)")

## 5 — Lags, promedios móviles, deltas e índices

Todo con `.over(KEYS)` y sobre el panel ordenado por mes, así cada operación se hace
**dentro de cada serie** y nunca mezcla productos.

| Familia | Cómo se calcula | Por qué |
|---|---|---|
| **lags** | `shift(k)` | la historia cruda |
| **promedios móviles** | `rolling_mean(w)`, incluyendo el mes actual | el nivel suavizado, sin el ruido de un mes |
| **deltas de share** | share − share de `t−k`, y share − su media móvil | **la redistribución**: no cuánto participa, sino cuánto *cambió* su participación |
| **índices** | `tn(t) / tn(t−1)`, recortado | el cambio en forma multiplicativa, que es como se mueve la demanda |

Los `rolling_mean` de polars **incluyen la fila actual**, que es lo correcto acá: el
promedio de los últimos 3 meses "a la fecha `t`" incluye a `t`, que ya pasó.

In [ ]:
t0 = time.time()

df = df.sort(KEYS + ["m"])

# ── Lags y promedios moviles de tn ───────────────────────────────────────
df = df.with_columns(
    *[pl.col("tn").shift(k).over(KEYS).alias(f"tn_lag{k}") for k in range(1, L + 1)],
    *[pl.col("tn").rolling_mean(w).over(KEYS).alias(f"tn_ma{w}")
      for w in PARAM['ventanas_ma']],
    *[pl.col("req_qty").rolling_mean(w).over(KEYS).alias(f"qty_ma{w}")
      for w in PARAM['ventanas_ma'][:2]],
    pl.col("req_qty").shift(1).over(KEYS).alias("qty_lag1"),
    pl.col("req_tn").rolling_mean(3).over(KEYS).alias("reqtn_ma3"),
    # maximo EXPANSIVO: el mayor hasta este mes, no el de toda la serie
    pl.col("tn").cum_max().over(KEYS).alias("tn_pico_hasta_aca"),
)

# ── Lags y medias moviles de cada share ─────────────────────────────────
exprs = []
for s in SHARES:
    exprs += [pl.col(s).shift(k).over(KEYS).alias(f"{s}_lag{k}")
              for k in range(1, PARAM['lags_share'] + 1)]
    exprs += [pl.col(s).rolling_mean(w).over(KEYS).alias(f"{s}_ma{w}")
              for w in (3, 6)]
df = df.with_columns(exprs)

# ── DELTAS de share: la redistribucion ──────────────────────────────────
# Dos versiones, y dicen cosas distintas:
#   _d1  : contra el mes anterior -> el movimiento reciente, mas ruidoso
#   _dma3: contra su propia media movil -> desvio respecto de su nivel habitual,
#          que es la senial de "algo cambio" y no de "siempre fue asi"
exprs = []
for s in SHARES:
    exprs += [
        (pl.col(s) - pl.col(f"{s}_lag1")).alias(f"{s}_d1"),
        (pl.col(s) - pl.col(f"{s}_ma3")).alias(f"{s}_dma3"),
    ]
    if PARAM['lags_share'] >= 3:
        exprs.append((pl.col(s) - pl.col(f"{s}_lag3")).alias(f"{s}_d3"))
df = df.with_columns(exprs)

# ── INDICES: ratio contra el mes anterior ───────────────────────────────
# Recortados a 'techo_indice': un producto que pasa de 0,01 a 10 tn da ratio 1000, y
# ese outlier no informa nada -- satura los cortes del arbol.
TECHO = PARAM['techo_indice']


def indice(num, den, nombre):
    return (pl.when(pl.col(den).abs() > 1e-9)
              .then((pl.col(num) / pl.col(den)).clip(0.0, TECHO))
              .otherwise(pl.lit(None, dtype=pl.Float64))
              .alias(nombre))


idx = [
    indice("tn", "tn_lag1", "idx_tn_mom"),          # mes contra mes anterior
    indice("tn", "tn_ma3", "idx_tn_vs_ma3"),        # mes contra su nivel reciente
    indice("tn", "tn_pico_hasta_aca", "idx_tn_vs_pico"),   # fraccion de su pico
    indice("req_qty", "qty_lag1", "idx_qty_mom"),   # "cuantas veces vendi" vs el mes pasado
]
if not ES_PC:
    idx.append(indice("n_clientes", "n_clientes", "idx_clientes_mom"))
df = df.with_columns(idx)

if not ES_PC:
    # a nivel producto si existe "cuantos clientes distintos me compraron"
    df = df.with_columns(pl.col("n_clientes").shift(1).over(KEYS).alias("n_clientes_lag1"))
    df = df.with_columns(indice("n_clientes", "n_clientes_lag1", "idx_clientes_mom"))

# ── Actividad reciente: cuantos de los ultimos 6 meses tuvo venta ────────
df = df.with_columns(
    (pl.col("tn") > 0).cast(pl.Int8).alias("vendio"),
)
df = df.with_columns(
    pl.col("vendio").rolling_mean(6).over(KEYS).alias("frac_meses_con_venta_6"),
    pl.col("vendio").rolling_sum(3).over(KEYS).alias("meses_con_venta_3"),
    (pl.col("periodo") % 100).alias("mes_del_anio"),
    (pl.col("edad").is_between(0, 6)).cast(pl.Int8).alias("es_nuevo"),
)

print(f"panel con features: {df.height:,} filas x {df.width} columnas")
print(f"[{time.time()-t0:.0f}s]")

## 6 — El target

La fila de `t` predice las toneladas de `t + horizonte`. `shift(-H)` sobre el panel
densificado y ordenado por mes da exactamente ese valor; en el borde final queda nulo,
y **ésas son las filas de inferencia**.

Se genera una sola `clase_tn`, en toneladas, sin normalizar. Eso simplifica todo el
camino: en `03_Optuna` se usa `target='clase_tn'`, la reconstrucción a toneladas es la
identidad, y no hace falta arrastrar `B0`/`B1` ni verificar ningún round-trip.

In [ ]:
df = df.sort(KEYS + ["m"])
df = df.with_columns(
    pl.col("tn").shift(-H).over(KEYS).alias("clase_tn"),
    # a que mes apunta la prediccion. Aritmetico, no shift: con shift queda nulo justo
    # en las filas de inferencia, que son las que necesitan saber su mes objetivo.
    ((((pl.col("m") + H - 1) // 12) * 100)
     + ((pl.col("m") + H - 1) % 12) + 1).alias("periodo_objetivo"),
)

_sup = int(df["clase_tn"].is_not_null().sum())
print(f"filas con target      : {_sup:,}")
print(f"filas de inferencia   : {df.height - _sup:,}")
print(f"periodos de inferencia: "
      f"{sorted(df.filter(pl.col('clase_tn').is_null())['periodo'].unique().to_list())}")

## 7 — Control de data leakage

Cinco chequeos. Si alguno falla, el notebook corta antes de exportar: un dataset con
leakage produce un WAPE hermoso en validación y basura en Kaggle, y es el error más
caro de todos porque no se nota hasta el final.

In [ ]:
errores = []


def chk(ok, msg):
    print(f"  [{'ok   ' if ok else 'ERROR'}] {msg}")
    if not ok:
        errores.append(msg)


print("CONTROL DE DATA LEAKAGE")
print("=" * 74)

# ── 1) El shift del target apunta donde debe ─────────────────────────────
# Se toma una serie con historia larga y se verifica fila por fila que
# clase_tn[i] == tn[i + H]. Es el chequeo mas directo y el que mas veces salva.
_una = (df.filter(pl.col("clase_tn").is_not_null())
          .group_by(KEYS).agg(pl.len().alias("n")).sort("n", descending=True).head(1))
_k = {c: _una[c][0] for c in KEYS}
_serie = df.filter(pl.all_horizontal([pl.col(c) == v for c, v in _k.items()])).sort("m")
_tn = _serie["tn"].to_list()
_cl = _serie["clase_tn"].to_list()
_malos = [i for i in range(len(_tn) - H)
          if _cl[i] is not None and abs(_cl[i] - _tn[i + H]) > 1e-9]
chk(not _malos, f"clase_tn[i] == tn[i+{H}] en la serie {_k} ({len(_tn)} meses, "
                f"{len(_malos)} discrepancias)")

# ── 2) Ninguna columna del futuro entre las features ─────────────────────
PROHIBIDAS = {"clase_tn", "periodo_objetivo", "m_muere", "m", "periodo"} | set(KEYS)
FEATURES = [c for c in df.columns if c not in PROHIBIDAS and c != "m_nace"]
chk(not (set(FEATURES) & {"clase_tn", "periodo_objetivo"}),
    "el target no esta entre las features")
chk("m_muere" not in FEATURES,
    "m_muere (ultimo mes con venta = dato del futuro) NO es feature")
chk("m_nace" not in FEATURES,
    "m_nace no es feature; la edad causal la reemplaza")

# ── 3) Ninguna feature es el target disfrazado ───────────────────────────
_mc = df.filter(pl.col("clase_tn").is_not_null())
if _mc.height > 300_000:
    _mc = _mc.sample(n=300_000, seed=PARAM['semilla'])
_y = _mc["clase_tn"].to_numpy().astype(np.float64)
_num = [c for c in FEATURES if _mc.schema[c] in
        (pl.Float32, pl.Float64, pl.Int8, pl.Int16, pl.Int32, pl.Int64,
         pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64)]
_sosp = []
for c in _num:
    x = _mc[c].to_numpy().astype(np.float64)
    ok = np.isfinite(x) & np.isfinite(_y)
    if ok.sum() < 100 or x[ok].std() == 0:
        continue
    r = float(np.corrcoef(x[ok], _y[ok])[0, 1])
    if abs(r) > 0.999:
        _sosp.append((c, round(r, 5)))
chk(not _sosp, f"ninguna de las {len(_num)} features numericas correlaciona >0.999 "
               f"con clase_tn (medido sobre {_mc.height:,} filas)  {_sosp}")

# ── 4) Los promedios moviles no miran adelante ───────────────────────────
# tn_ma3 en el mes t tiene que ser el promedio de t-2, t-1, t. Se verifica a mano.
_s2 = _serie.select("m", "tn", "tn_ma3").sort("m")
_tn2, _ma = _s2["tn"].to_list(), _s2["tn_ma3"].to_list()
_err_ma = max((abs(_ma[i] - np.mean(_tn2[i-2:i+1]))
               for i in range(2, len(_tn2)) if _ma[i] is not None), default=0.0)
chk(_err_ma < 1e-9, f"tn_ma3[t] == promedio(tn[t-2..t]): error maximo {_err_ma:.2e}")

# ── 5) Los shares no usan el futuro: el denominador es del mismo mes ─────
# Se recalcula un share a mano para una fila y se compara.
if 'cat3' in PARAM['niveles_share']:
    _f = _serie.filter(pl.col("tn") > 0).head(1)
    if _f.height:
        _mm, _c3 = int(_f["m"][0]), _f["cat3"][0]
        _tot = (df.filter((pl.col("m") == _mm) & (pl.col("cat3") == _c3))
                  .unique(subset=["product_id"])["tn_prod"].sum())
        _esp = float(_f["tn_prod"][0]) / _tot if _tot else 0.0
        _obs = float(_f["sh_prod_en_cat3"][0])
        chk(abs(_esp - _obs) < 1e-6,
            f"sh_prod_en_cat3 recalculado a mano coincide "
            f"(esperado {_esp:.6f}, observado {_obs:.6f})")

print("=" * 74)
if errores:
    raise RuntimeError(f"Control de leakage FALLIDO: {errores}")
print(f"Control superado. {len(FEATURES)} features.")

## 8 — Exportar

Las features van en **Float32**: LightGBM discretiza cada una en 255 bins antes de
entrenar, así que la precisión de Float64 se descarta igual y el dataset ocupa la mitad.
`clase_tn` y `tn0` quedan en Float64, que son las que entran en el cálculo del WAPE.

La columna del mes actual se exporta como **`tn0`** — el nombre que usa el pipe
original — para que el baseline naive de `03_Optuna` («repetir lo del mes `t`») funcione
sin tocar nada.

In [ ]:
CTX_F64 = {"clase_tn", "tn0"}

salida = df.rename({"tn": "tn0"}).drop(["m_nace", "m_muere"])
# La lista de features se armo antes del rename, asi que decia 'tn'. Se corrige para
# que el features.json describa el parquet de verdad.
FEATURES = ["tn0" if c == "tn" else c for c in FEATURES]
_f64 = [c for c, t in salida.schema.items()
        if t == pl.Float64 and c not in CTX_F64]
salida = salida.with_columns([pl.col(c).cast(pl.Float32) for c in _f64])

path_out = RUTA_FE / NOMBRE
salida.write_parquet(path_out)

print(f"Guardado: {path_out}")
print(f"  {salida.height:,} filas x {salida.width} columnas")
print(f"  Float64 -> Float32: {len(_f64)} columnas (quedan en Float64: {sorted(CTX_F64)})")
print(f"  tamanio en disco: {path_out.stat().st_size / 1e6:.0f} MB")
print(f"  RAM estimada    : {salida.estimated_size() / 1e9:.2f} GB")

# ── Resumen de las familias de features ─────────────────────────────────
familias = {
    "shares": [c for c in FEATURES if c.startswith("sh_") and "_d" not in c and "_ma" not in c and "_lag" not in c],
    "lags de share": [c for c in FEATURES if c.startswith("sh_") and "_lag" in c],
    "ma de share": [c for c in FEATURES if c.startswith("sh_") and "_ma" in c],
    "deltas de share": [c for c in FEATURES if c.startswith("sh_") and ("_d1" in c or "_d3" in c or "_dma" in c)],
    "lags de tn": [c for c in FEATURES if c.startswith("tn_lag")],
    "ma de tn/qty": [c for c in FEATURES if "_ma" in c and not c.startswith("sh_")],
    "indices": [c for c in FEATURES if c.startswith("idx_")],
    "totales/contexto": [c for c in FEATURES if c.startswith(("tn_prod", "tn_cli", "tn_cat", "tn_mercado", "n_"))],
    "calendario/ciclo": [c for c in FEATURES if c in ("mes_del_anio", "edad", "es_nuevo",
                                                      "vendio", "frac_meses_con_venta_6",
                                                      "meses_con_venta_3", "tn_pico_hasta_aca")],
}
print("\nFamilias de features:")
_vistas = set()
for k, v in familias.items():
    if v:
        print(f"  {k:20s} {len(v):3d}")
        _vistas |= set(v)
_resto = [c for c in FEATURES if c not in _vistas]
if _resto:
    print(f"  {'otras':20s} {len(_resto):3d}   {_resto[:8]}")

with open(RUTA_FE / NOMBRE.replace(".parquet", "_features.json"), "w", encoding="utf-8") as f:
    json.dump({"nombre": NOMBRE, "param": PARAM, "n_features": len(FEATURES),
               "features": FEATURES, "shares": SHARES,
               "familias": {k: v for k, v in familias.items() if v}},
              f, indent=2, ensure_ascii=False, default=str)

## 9 — Cómo usarlo en `03_Optuna`

```python
PARAM['dataset_fe'] = '<el nombre que imprimio arriba>'
PARAM['target']     = 'clase_tn'     # OBLIGATORIO: pipe_2 no genera las versiones
                                     # normalizadas, y con 'clase_tn' la
                                     # reconstruccion a toneladas es la identidad
PARAM['sufijo']     = 'fe2'          # para no pisar los experimentos del pipe original
```

Con `target='clase_tn'` el `TARGET_KIND` es `'nivel'`, así que `reconstruir_nivel`
devuelve la predicción tal cual y el chequeo de round-trip da 0 exacto. No hacen falta
`B0`/`B1`.

### El objetivo y el techo de árboles importan mucho acá

Medido sobre una muestra de 40 productos (353.908 filas de train, validación en
201907-201908), prediciendo `clase_tn` en toneladas:

| | WAPE val |
|---|---|
| naive (repetir `tn0`) | 0,2566 |
| media móvil de 3 meses | 0,2169 |
| **`regression` (L2), 400 árboles** | **0,1573** |
| `regression` (L2), 60 árboles | 0,1756 |
| `regression_l1`, 400 árboles | 0,3999 |
| `tweedie`, 400 árboles | 0,2078 |

Dos conclusiones prácticas:

- **Usá `objective_lgbm='regression'`** (L2). Con el target en toneladas crudas,
  `regression_l1` es 2,5 veces peor: minimiza el error absoluto **por fila**, así que
  subestima sistemáticamente las series grandes — y el WAPE, que pondera por volumen,
  castiga exactamente eso. `tweedie` tampoco compensa acá.
- **No bajes `techo_arboles` de 400.** Con 60 el modelo pierde 0,018 de WAPE, y si
  además Optuna sortea un `learning_rate` bajo no llega a aprender nada: con 3 trials y
  techo 60 este mismo dataset dio 0,486, *peor que el naive*. No era el dataset, era el
  presupuesto.

### Una advertencia sobre el target

pipe_2 predice **toneladas crudas**, mientras el pipe original predice un target
normalizado y después reconstruye. Predecir el nivel es más difícil: el modelo tiene que
aprender la escala de cada serie además de su forma.

Por eso el uso más prometedor de pipe_2 **no** es como reemplazo, sino **joinear estas
features al dataset del pipe original**, que sí trae los targets normalizados:

```python
fe1 = pl.read_parquet(RUTA_FE / "preprocesado_..._24lags_recta_2deltas.parquet")
fe2 = pl.read_parquet(RUTA_FE / "preprocesado_..._share_2h_fe2.parquet")

nuevas = [c for c in fe2.columns
          if c.startswith(("sh_", "idx_")) or c in ("frac_meses_con_venta_6",)]
juntos = fe1.join(fe2.select(["product_id", "customer_id", "periodo"] + nuevas),
                  on=["product_id", "customer_id", "periodo"], how="left")
```

Ahí tenés los shares con el target normalizado, que es la combinación que más promete.

## Qué comparar en el leaderboard

Los dos pipes escriben en el mismo `exp/leaderboard.csv`, así que la comparación es
directa. Y es una comparación de **familias de features**, no de modelos:

| | features | qué prueba |
|---|---|---|
| pipe original | ~616: 24 lags en 5 niveles de agregación | que más historia propia ayuda |
| **pipe_2** | ~120: 12 lags + shares, deltas de share e índices | que el **contexto competitivo** ayuda más que la historia larga |

Si pipe_2 gana con una quinta parte de las columnas, la conclusión es que el problema no
era falta de historia sino falta de contexto. Y si pierde, también aprendiste algo: la
serie propia tenía más información que la posición relativa.

Lo más interesante sería un tercer experimento con **las dos familias juntas**, que se
arma con un `join` de los dos parquet por las claves y el período.